In [ ]:
# ── Cell 1 — Imports + Config ─────────────────────────────────────────────────

import os
import random
import numpy as np
import torch
import torch.nn as nn
import tifffile
import albumentations as A
import torchvision.models as tvm

from pathlib import Path
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision.models import ResNet34_Weights, ResNet18_Weights

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ── Device ────────────────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

# ── Paths ─────────────────────────────────────────────────────────────────────
BASE_DIR = Path("/kaggle/input/datasets/rohitrobert34/galaxyeye-change-detection")

PATHS = {
    "train": {
        "pre":    BASE_DIR / "train" / "pre-event",
        "post":   BASE_DIR / "train" / "post-event",
        "target": BASE_DIR / "train" / "target",
    },
    "val": {
        "pre":    BASE_DIR / "change_detection_assignment_data" / "val" / "pre-event",
        "post":   BASE_DIR / "change_detection_assignment_data" / "val" / "post-event",
        "target": BASE_DIR / "change_detection_assignment_data" / "val" / "target",
    },
    "test": {
        "pre":    BASE_DIR / "change_detection_assignment_data" / "test" / "pre-event",
        "post":   BASE_DIR / "change_detection_assignment_data" / "test" / "post-event",
        "target": BASE_DIR / "change_detection_assignment_data" / "test" / "target",
    },
}

FILES = {}
for split in ["train", "val", "test"]:
    FILES[split] = sorted([p.name for p in PATHS[split]["pre"].glob("*.tif")])

for split in ["train", "val", "test"]:
    pre_n    = len(FILES[split])
    post_n   = len(list(PATHS[split]["post"].glob("*.tif")))
    target_n = len(list(PATHS[split]["target"].glob("*.tif")))
    print(f"{split}: pre={pre_n}, post={post_n}, target={target_n}")

# ── Architecture constants ────────────────────────────────────────────────────
IN_CH_EO    = 3    # RGB pre-event → ResNet34
IN_CH_SAR   = 6    # [SAR, eo_lum, cross, cross_abs, ndvi, sar_ndvi] → ResNet18
NUM_CLASSES = 1

# ── Training hyperparameters ──────────────────────────────────────────────────
BATCH_SIZE   = 4      # NEW: gradient accumulation → effective batch = 4 * 2 
ACCUM_STEPS = 2
NUM_EPOCHS   = 60
ENCODER_LR   = 1e-5    # both encoders (low — pretrained)
DECODER_LR   = 1e-3    # decoder + head (high — random init)
WEIGHT_DECAY = 3e-4
T_MAX        = 50
ETA_MIN      = 1e-6
PATIENCE     = 10
POS_WEIGHT   = 3.0
NUM_WORKERS  = 2
CROP_SIZE    = 512
CHECKPOINT   = "best_v9(4b - 8 batches).pth"
THRESHOLD    = 0.4

# ── Loss hyperparameters ──────────────────────────────────────────────────────
# γ = 1.33 — CORRECTED from the V1–V4 bug where γ=0.75 inverted the focal mechanism
# Per Abraham & Khan 2019: exponent > 1 focuses on hard examples
FOCAL_ALPHA   = 0.2   # FP penalty — low (false alarms acceptable)
FOCAL_BETA    = 0.8   # FN penalty — high (missing damage is critical)
FOCAL_GAMMA   = 1.5  # ← CORRECTED (was 0.75 — that mathematically inverted focalization)
FOCAL_SMOOTH  = 1.0
LOSS_FT_W     = 1.0  # weight for FocalTversky in combined loss
LOSS_DICE_W   = 0.0  # weight for Dice in combined loss

print("\nAll config loaded ✓")
print(f"EO channels: {IN_CH_EO} | SAR channels: {IN_CH_SAR}")
print(f"Batch: {BATCH_SIZE} | Epochs: {NUM_EPOCHS} | γ: {FOCAL_GAMMA}")

In [ ]:
# ── Cell 2 — DualChangeDataset ────────────────────────────────────────────────
#
# Returns THREE items per __getitem__:
#   eo_tensor  : (3, H, W)  float32 — pre-event RGB, normalized
#   sar_tensor : (6, H, W)  float32 — SAR-side engineered channels
#   mask       : (H, W)     float32 — binary change mask {0, 1}
#
# Augmentation contract:
#   • Both EO and SAR images must receive IDENTICAL spatial transforms
#     (same crop coordinates, same flip directions) so mask alignment is preserved.
#   • Albumentations handles this via its 'additional_targets' API:
#     pass sar_image as an extra image so A.Compose applies the same random
#     spatial ops to both.
#   • NEVER apply colour augmentation to SAR channels (hard rule #6).

class DualChangeDataset(Dataset):

    def __init__(self, split, paths, file_names, transform=None):
        self.split      = split
        self.paths      = paths
        self.file_names = file_names
        self.transform  = transform   # albumentations Compose

    def __len__(self):
        return len(self.file_names)

    # ── Step 1: Load raw files ─────────────────────────────────────────────────
    def _load_triplet(self, idx):
        fname = self.file_names[idx]

        pre  = tifffile.imread(self.paths[self.split]["pre"]    / fname)  # (H,W,3) uint8
        post = tifffile.imread(self.paths[self.split]["post"]   / fname)  # (H,W)   uint8
        mask = tifffile.imread(self.paths[self.split]["target"] / fname)  # (H,W)   uint8

        post = post[:, :, np.newaxis]                              # (H,W,1)
        unique_vals = np.unique(mask)
        if np.any(unique_vals >= 2):
            mask = np.where(mask >= 2, 1, 0).astype(np.uint8)
        else:
            mask = np.where(mask >= 1, 1, 0).astype(np.uint8)        # remap: 0/1

        return pre, post, mask, fname

    # ── Step 2: Engineer features, split into EO and SAR stacks ───────────────
    def _engineer_features(self, pre, post):
        # Normalise FIRST — never compute on raw uint8
        pre_norm  = pre.astype(np.float32) / 255.0              # (H,W,3) [0,1]
        post_norm = (post.astype(np.float32) - 1.0) / 230.0     # (H,W,1) [0,1]
        post_norm = np.clip(post_norm, 0.0, 1.0)

        # ── EO luminance — perceptual brightness ──────────────────────────────
        eo_lum = (
            0.299 * pre_norm[:, :, 0:1] +
            0.587 * pre_norm[:, :, 1:2] +
            0.114 * pre_norm[:, :, 2:3]
        )                                                        # (H,W,1) [0,1]

        # ── Cross-modal disagreement ───────────────────────────────────────────
        cross     = post_norm - eo_lum                           # (H,W,1) [-1,1]
        cross_abs = np.abs(cross)                                # (H,W,1) [0,1]

        # ── NDVI (Normalised Green-Red Difference Index) ───────────────────────
        # Uses Green (ch1) and Red (ch0) from EO
        # Damaged vegetation has dramatically lower NDVI
        ndvi = (
            (pre_norm[:, :, 1:2] - pre_norm[:, :, 0:1]) /
            (pre_norm[:, :, 1:2] + pre_norm[:, :, 0:1] + 1e-8)
        )                                                        # (H,W,1) [-1,1]

        # ── SAR-NDVI interaction ───────────────────────────────────────────────
        # High SAR backscatter + low NDVI → strong destruction signature
        sar_ndvi = post_norm * (1.0 - np.clip(ndvi, 0.0, 1.0))  # (H,W,1) [0,1]

        # ── EO stack: 3 channels (goes into ResNet34 encoder) ─────────────────
        eo_stack  = pre_norm                                     # (H,W,3)

        # ── SAR stack: 6 channels (goes into patched ResNet18 encoder) ────────
        sar_stack = np.concatenate(
            [post_norm, eo_lum, cross, cross_abs, ndvi, sar_ndvi],
            axis=-1
        ).astype(np.float32)                                     # (H,W,6)

        return eo_stack, sar_stack

    # ── Step 3: Augment + convert ─────────────────────────────────────────────
    def __getitem__(self, idx):
        pre, post, mask, fname = self._load_triplet(idx)
        eo_stack, sar_stack   = self._engineer_features(pre, post)

        if self.transform is not None:
            # Pass sar_stack as 'image2' — Compose applies identical spatial ops
            augmented = self.transform(
                image=eo_stack,
                image2=sar_stack,
                mask=mask
            )
            eo_stack  = augmented["image"]    # (H,W,3)
            sar_stack = augmented["image2"]   # (H,W,6)
            mask      = augmented["mask"]     # (H,W)

        # Transpose from HWC → CHW for PyTorch
        eo_tensor  = torch.from_numpy(np.transpose(eo_stack,  (2, 0, 1))).float()  # (3,H,W)
        sar_tensor = torch.from_numpy(np.transpose(sar_stack, (2, 0, 1))).float()  # (6,H,W)
        mask_tensor = torch.from_numpy(mask).float()                                # (H,W)

        return eo_tensor, sar_tensor, mask_tensor, fname


# ── Augmentation pipelines ────────────────────────────────────────────────────
# additional_targets tells Albumentations: treat 'image2' exactly like 'image'
# (same spatial transforms, but image2 is NOT subject to colour jitter if we add it)

train_transform = A.Compose([
    A.RandomCrop(CROP_SIZE, CROP_SIZE, p=1.0),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
], additional_targets={"image2": "image"},   # ← key: syncs SAR transforms
   is_check_shapes=False)

val_test_transform = A.Compose([
    A.CenterCrop(CROP_SIZE, CROP_SIZE, p=1.0),
], additional_targets={"image2": "image"},
   is_check_shapes=False)

print("DualChangeDataset defined ✓")
print(f"EO stack shape:  (H, W, {IN_CH_EO})")
print(f"SAR stack shape: (H, W, {IN_CH_SAR})")

In [ ]:
# ── Cell 3 — Sampler + DataLoaders ───────────────────────────────────────────

def compute_weights(target_dir, file_names, pos_weight=POS_WEIGHT):
    weights = []
    for fname in file_names:
        mask = tifffile.imread(target_dir / fname)
        mask = np.where(mask >= 2, 1, 0).astype(np.uint8)
        weights.append(pos_weight if np.any(mask == 1) else 1.0)
    return np.array(weights, dtype=np.float32)


train_weights = compute_weights(PATHS["train"]["target"], FILES["train"])
num_pos = int(np.sum(train_weights > 1.0))
num_neg = int(np.sum(train_weights == 1.0))
print(f"Tiles with change:    {num_pos}")
print(f"Tiles without change: {num_neg}")

train_sampler = WeightedRandomSampler(
    weights=torch.tensor(train_weights, dtype=torch.float32),
    num_samples=len(train_weights),
    replacement=True
)
print("Sampler ready ✓")

# ── Datasets ──────────────────────────────────────────────────────────────────
train_dataset = DualChangeDataset("train", PATHS, FILES["train"], train_transform)
val_dataset   = DualChangeDataset("val",   PATHS, FILES["val"],   val_test_transform)
test_dataset  = DualChangeDataset("test",  PATHS, FILES["test"],  val_test_transform)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

# ── DataLoaders ───────────────────────────────────────────────────────────────
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE,
    sampler=train_sampler,          # shuffle=True is illegal with sampler= (hard rule #5)
    num_workers=NUM_WORKERS, pin_memory=True
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE,
    shuffle=False, num_workers=NUM_WORKERS, pin_memory=True
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE,
    shuffle=False, num_workers=NUM_WORKERS, pin_memory=True
)

print(f"Train batches: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}")

# ── Batch sanity check ────────────────────────────────────────────────────────
eo_b, sar_b, mask_b, fnames_b = next(iter(train_loader))
print(f"\nBatch check:")
print(f"  EO   shape: {eo_b.shape}")    # (B, 3, 512, 512)
print(f"  SAR  shape: {sar_b.shape}")   # (B, 6, 512, 512)
print(f"  Mask shape: {mask_b.shape}")  # (B, 512, 512)
print(f"  EO   range: [{eo_b.min():.3f}, {eo_b.max():.3f}]")
print(f"  SAR  range: [{sar_b.min():.3f}, {sar_b.max():.3f}]")
print(f"  Mask unique: {torch.unique(mask_b)}")

In [ ]:
# ── Cell 4 — CBAM + DualEncoderUNet ──────────────────────────────────────────
#
# CBAM = Convolutional Block Attention Module (Woo et al. 2018)
# Two sub-modules applied sequentially:
#   1. ChannelAttention — "what" to focus on (which feature maps matter)
#   2. SpatialAttention — "where" to focus on (which spatial locations matter)
#
# Placement per plan:
#   • After bottleneck concat (EO + SAR features combined)
#   • After skip-fusion in d3 and d2 decoder stages
#   • NOT everywhere — plan explicitly warns against that

class ChannelAttention(nn.Module):
    """
    Squeezes spatial dims → MLP → sigmoid gate applied to channels.
    reduction=16 means the MLP bottleneck is channels//16 wide.
    """
    def __init__(self, channels, reduction=16):
        super().__init__()
        # Both avg and max pool independently summarize the feature map
        self.avg_pool = nn.AdaptiveAvgPool2d(1)   # (B, C, 1, 1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)   # (B, C, 1, 1)

        # Shared MLP: C → C//reduction → C
        mid = max(channels // reduction, 1)
        self.mlp = nn.Sequential(
            nn.Flatten(),
            nn.Linear(channels, mid, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(mid, channels, bias=False),
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        B, C, H, W = x.shape
        avg_out = self.mlp(self.avg_pool(x))   # (B, C)
        max_out = self.mlp(self.max_pool(x))   # (B, C)
        gate    = self.sigmoid(avg_out + max_out).view(B, C, 1, 1)
        return x * gate                         # (B, C, H, W)


class SpatialAttention(nn.Module):
    """
    Compresses channel dim → conv → sigmoid gate applied spatially.
    kernel=7 gives a wide receptive field for locating changed regions.
    """
    def __init__(self, kernel_size=7):
        super().__init__()
        pad = kernel_size // 2
        self.conv    = nn.Conv2d(2, 1, kernel_size=kernel_size,
                                 padding=pad, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # Summarize channels with avg and max → (B, 2, H, W)
        avg_out = torch.mean(x, dim=1, keepdim=True)   # (B, 1, H, W)
        max_out, _ = torch.max(x, dim=1, keepdim=True)  # (B, 1, H, W)
        combined = torch.cat([avg_out, max_out], dim=1)  # (B, 2, H, W)

        gate = self.sigmoid(self.conv(combined))          # (B, 1, H, W)
        return x * gate                                   # (B, C, H, W)


class CBAM(nn.Module):
    """Full CBAM: channel attention → spatial attention."""
    def __init__(self, channels, reduction=16, spatial_kernel=7):
        super().__init__()
        self.ca = ChannelAttention(channels, reduction)
        self.sa = SpatialAttention(spatial_kernel)

    def forward(self, x):
        x = self.ca(x)
        x = self.sa(x)
        return x


# ── ConvBlock and UpBlock ─────────────────────────────────────────────────────

class ConvBlock(nn.Module):
    """Two 3×3 conv → BN → ReLU layers. Standard UNet decoder block."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class FusionUpBlock(nn.Module):
    """
    One decoder stage for the dual-encoder model.

    Inputs:
        x        : features from the previous (deeper) decoder stage   (B, in_ch, H, W)
        eo_skip  : skip from EO encoder at this scale                   (B, eo_skip_ch, 2H, 2W)
        sar_skip : skip from SAR encoder at this scale                  (B, sar_skip_ch, 2H, 2W)

    Steps:
        1. Upsample x by 2× with transposed conv → (B, in_ch//2, 2H, 2W)
        2. Concatenate with eo_skip + sar_skip along channel dim
        3. Run ConvBlock to mix everything → (B, out_ch, 2H, 2W)
        4. Optionally apply CBAM for attention-refined output

    use_cbam=True for d3, d2 (higher-resolution stages where boundary
    recovery matters most, per plan).
    """
    def __init__(self, in_ch, eo_skip_ch, sar_skip_ch, out_ch, use_cbam=False):
        super().__init__()
        self.up   = nn.ConvTranspose2d(in_ch, in_ch // 2, kernel_size=2, stride=2)
        # After concat: (in_ch//2) + eo_skip_ch + sar_skip_ch
        self.conv = ConvBlock(in_ch // 2 + eo_skip_ch + sar_skip_ch, out_ch)
        self.cbam = CBAM(out_ch) if use_cbam else nn.Identity()

    def forward(self, x, eo_skip, sar_skip):
        x = self.up(x)                                      # upsample
        x = torch.cat([x, eo_skip, sar_skip], dim=1)       # fuse
        x = self.conv(x)                                    # mix
        x = self.cbam(x)                                    # attend (or identity)
        return x


# ── DualEncoderUNet ───────────────────────────────────────────────────────────

class DualEncoderUNet(nn.Module):
    """
    Architecture:
      EO  branch (ResNet34): e0→e4  channels [64, 64, 128, 256, 512]
      SAR branch (ResNet18): s0→s4  channels [64, 64, 128, 256, 512]

      Bottleneck: cat(e4, s4) → (B,1024,H/32,W/32) → CBAM → bottleneck

      Decoder (FusionUpBlock, skip = eo_skip + sar_skip):
        d3: 1024 → 256  (CBAM on)
        d2:  256 → 128  (CBAM on)
        d1:  128 →  64  (no CBAM — lower res, less boundary work needed)
        d0:   64 →  32  (no CBAM)

      Final: upsample → Conv(32,1,1) → raw logits (B,1,H,W)
    """

    def __init__(self, in_ch_eo=IN_CH_EO, in_ch_sar=IN_CH_SAR, num_classes=NUM_CLASSES):
        super().__init__()

        # ── EO Encoder — ResNet34, pretrained ImageNet ────────────────────────
        eo_enc = tvm.resnet34(weights=ResNet34_Weights.DEFAULT)
        # conv1 already expects 3 channels — no patching needed for EO
        self.eo_e0   = nn.Sequential(eo_enc.conv1, eo_enc.bn1, eo_enc.relu)  # (B,64,H/2,W/2)
        self.eo_pool = eo_enc.maxpool
        self.eo_e1   = eo_enc.layer1   # (B, 64,  H/4,  W/4)
        self.eo_e2   = eo_enc.layer2   # (B, 128, H/8,  W/8)
        self.eo_e3   = eo_enc.layer3   # (B, 256, H/16, W/16)
        self.eo_e4   = eo_enc.layer4   # (B, 512, H/32, W/32)

        # ── SAR Encoder — ResNet34, pretrained, patched for in_ch_sar ─────────
        sar_enc = tvm.resnet34(weights=ResNet34_Weights.DEFAULT)

        # Patch conv1: original expects 3ch, we need in_ch_sar channels.
        old_w = sar_enc.conv1.weight.data  # (64, 3, 7, 7)
        new_conv = nn.Conv2d(in_ch_sar, 64, kernel_size=7,
                            stride=2, padding=3, bias=False)
        with torch.no_grad():
             # Repeat pretrained weights along channel dim, then trim and rescale
             repeated = old_w.repeat(1, in_ch_sar // 3 + 1, 1, 1)[:, :in_ch_sar]
             new_conv.weight.data = repeated / (in_ch_sar / 3)
        sar_enc.conv1 = new_conv
 
        self.sar_e0 = nn.Sequential(sar_enc.conv1, sar_enc.bn1, sar_enc.relu)
        self.sar_pool = sar_enc.maxpool
        self.sar_e1 = sar_enc.layer1  # (B, 64, H/4, W/4)
        self.sar_e2 = sar_enc.layer2  # (B, 128, H/8, W/8)
        self.sar_e3 = sar_enc.layer3  # (B, 256, H/16, W/16)
        self.sar_e4 = sar_enc.layer4  # (B, 512, H/32, W/32)
        # ── Bottleneck attention (after cat of e4 + s4 → 1024ch) ─────────────
        self.bottleneck_cbam = CBAM(1024)

        # ── Decoder ───────────────────────────────────────────────────────────
        # FusionUpBlock(in_ch, eo_skip_ch, sar_skip_ch, out_ch, use_cbam)
        self.d3 = FusionUpBlock(1024, 256, 256, 256, use_cbam=True)   # CBAM on
        self.d2 = FusionUpBlock( 256, 128, 128, 128, use_cbam=True)   # CBAM on
        self.d1 = FusionUpBlock( 128,  64,  64,  64, use_cbam=False)
        self.d0 = FusionUpBlock(  64,  64,  64,  32, use_cbam=False)

        self.up_final = nn.ConvTranspose2d(32, 32, kernel_size=2, stride=2)
        self.head     = nn.Conv2d(32, num_classes, kernel_size=1)

    def forward(self, eo, sar):
        # ── EO encoder pass ───────────────────────────────────────────────────
        eo_s0 = self.eo_e0(eo)                   # (B, 64,  H/2,  W/2)
        eo_s1 = self.eo_e1(self.eo_pool(eo_s0))  # (B, 64,  H/4,  W/4)
        eo_s2 = self.eo_e2(eo_s1)                # (B, 128, H/8,  W/8)
        eo_s3 = self.eo_e3(eo_s2)                # (B, 256, H/16, W/16)
        eo_b  = self.eo_e4(eo_s3)                # (B, 512, H/32, W/32)

        # ── SAR encoder pass ──────────────────────────────────────────────────
        sar_s0 = self.sar_e0(sar)                    # (B, 64,  H/2,  W/2)
        sar_s1 = self.sar_e1(self.sar_pool(sar_s0))  # (B, 64,  H/4,  W/4)
        sar_s2 = self.sar_e2(sar_s1)                 # (B, 128, H/8,  W/8)
        sar_s3 = self.sar_e3(sar_s2)                 # (B, 256, H/16, W/16)
        sar_b  = self.sar_e4(sar_s3)                 # (B, 512, H/32, W/32)

        # ── Bottleneck fusion ─────────────────────────────────────────────────
        bottleneck = torch.cat([eo_b, sar_b], dim=1)   # (B, 1024, H/32, W/32)
        bottleneck = self.bottleneck_cbam(bottleneck)   # CBAM cross-modal attention

        # ── Decoder (each stage fuses both encoder skips) ─────────────────────
        x = self.d3(bottleneck, eo_s3, sar_s3)         # (B, 256, H/16, W/16)
        x = self.d2(x,          eo_s2, sar_s2)         # (B, 128, H/8,  W/8)
        x = self.d1(x,          eo_s1, sar_s1)         # (B, 64,  H/4,  W/4)
        x = self.d0(x,          eo_s0, sar_s0)         # (B, 32,  H/2,  W/2)

        x = self.up_final(x)                            # (B, 32,  H,    W)
        return self.head(x)                             # (B, 1,   H,    W) raw logits


# ── Instantiate + shape check ─────────────────────────────────────────────────
model = DualEncoderUNet(IN_CH_EO, IN_CH_SAR, NUM_CLASSES).to(DEVICE)

model.eval()
with torch.no_grad():
    dummy_eo  = torch.zeros(1, IN_CH_EO,  512, 512).to(DEVICE)
    dummy_sar = torch.zeros(1, IN_CH_SAR, 512, 512).to(DEVICE)
    out = model(dummy_eo, dummy_sar)

print(f"EO input:  {dummy_eo.shape}")   # (1, 3, 512, 512)
print(f"SAR input: {dummy_sar.shape}")  # (1, 6, 512, 512)
print(f"Output:    {out.shape}")         # (1, 1, 512, 512)
print("Model ready ✓")

In [ ]:
# ── Cell 5 — Loss Functions ───────────────────────────────────────────────────

class FocalTverskyLoss(nn.Module):
    """
    Tversky index with focal weighting.

    α = 0.3  → FP penalty (false alarms acceptable)
    β = 0.7  → FN penalty (missing damage is critical)
    γ = 1.33 → CORRECTED focal exponent (> 1 focuses on hard examples)
               V1-V4 used γ=0.75 which INVERTED the focal mechanism.
    """
    def __init__(self, alpha=FOCAL_ALPHA, beta=FOCAL_BETA,
                 gamma=FOCAL_GAMMA, smooth=FOCAL_SMOOTH):
        super().__init__()
        self.alpha  = alpha
        self.beta   = beta
        self.gamma  = gamma
        self.smooth = smooth

    def forward(self, pred, target):
        pred   = torch.sigmoid(pred)
        pred   = pred.view(-1)
        target = target.view(-1).float()

        tp = (pred * target).sum()
        fp = ((1 - target) * pred).sum()
        fn = (target * (1 - pred)).sum()

        tversky = (tp + self.smooth) / (
            tp + self.alpha * fp + self.beta * fn + self.smooth
        )
        return (1 - tversky) ** self.gamma


class FocalBCELoss(nn.Module):
    """
    Focal Binary Cross-Entropy for class imbalance.
    gamma > 1 focuses on hard examples; alpha balances positives/negatives.
    """
    def __init__(self, alpha=0.75, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, pred, target):
        # pred: (B,1,H,W) logits
        pred = torch.sigmoid(pred)
        pred = pred.view(-1)
        target = target.view(-1).float()

        eps = 1e-8
        p_t = pred * target + (1 - pred) * (1 - target)
        alpha_t = self.alpha * target + (1 - self.alpha) * (1 - target)

        focal_weight = alpha_t * (1 - p_t).pow(self.gamma)
        bce = -(target * torch.log(pred + eps) +
                (1 - target) * torch.log(1 - pred + eps))

        loss = (focal_weight * bce).mean()
        return loss


class DiceLoss(nn.Module):
    """
    Standard soft Dice loss — overlap-based, scale-invariant.
    Added for combined loss stability per plan (0.5 FT + 0.5 Dice).
    """
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, pred, target):
        pred   = torch.sigmoid(pred).view(-1)
        target = target.view(-1).float()

        intersection = (pred * target).sum()
        dice = (2 * intersection + self.smooth) / (
            pred.sum() + target.sum() + self.smooth
        )
        return 1 - dice


class CombinedLoss(nn.Module):
    """0.5 × FocalTversky + 0.5 × Dice (weights per plan)."""
    def __init__(self, ft_weight=LOSS_FT_W, dice_weight=LOSS_DICE_W):
        super().__init__()
        self.ft_weight   = ft_weight
        self.dice_weight = dice_weight
        self.ft          = FocalTverskyLoss()
        self.dice        = DiceLoss()

    def forward(self, pred, target):
        return self.ft_weight * self.ft(pred, target) + \
               self.dice_weight * self.dice(pred, target)


class HybridLoss(nn.Module):
    """
    Hybrid loss: lambda * Focal BCE + (1 - lambda) * Focal Tversky.
    """
    def __init__(self, lambda_focal_bce=0.3):
        super().__init__()
        self.lambda_fbce = lambda_focal_bce
        self.fbce = FocalBCELoss(alpha=0.75, gamma=2.0)
        self.ft = FocalTverskyLoss(
            alpha=FOCAL_ALPHA,
            beta=FOCAL_BETA,
            gamma=FOCAL_GAMMA,
            smooth=FOCAL_SMOOTH,
        )

    def forward(self, pred, target):
        fbce_loss = self.fbce(pred, target)
        ft_loss   = self.ft(pred, target)
        return self.lambda_fbce * fbce_loss + (1.0 - self.lambda_fbce) * ft_loss


#loss_fn = CombinedLoss() # old setup (symmetric FT + Dice)

#loss_fn = FocalTverskyLoss(
    #alpha=FOCAL_ALPHA,
    #beta=FOCAL_BETA,
    #gamma=FOCAL_GAMMA,
    #smooth=FOCAL_SMOOTH,
#)
loss_fn = HybridLoss(lambda_focal_bce=0.3)


# ── Sanity checks ─────────────────────────────────────────────────────────────
good_bg   = loss_fn(torch.full((1,1,4,4), -4.0), torch.zeros(1,4,4))
good_fg   = loss_fn(torch.full((1,1,4,4),  4.0), torch.ones(1,4,4))
torch.manual_seed(SEED)
logits_m  = torch.randn(1,1,4,4, requires_grad=True)
mask_m    = torch.randint(0, 2, (1,4,4)).float()
loss_m    = loss_fn(logits_m, mask_m)
loss_m.backward()

print(f"Loss (good bg pred):     {good_bg.item():.4f}")
print(f"Loss (good fg pred):     {good_fg.item():.4f}")
print(f"Loss (mixed):            {loss_m.item():.4f}")
print(f"Gradients finite:        {torch.isfinite(logits_m.grad).all().item()}")
print(f"γ = {FOCAL_GAMMA} (corrected) ✓")
print("Loss ready ✓")

In [ ]:
# ── Cell 6 — Optimizer + Scheduler + Early Stopping ─────────────────────────

# All encoder params (both EO and SAR encoders) → low LR (pretrained)
encoder_params = (
    list(model.eo_e0.parameters()) + list(model.eo_e1.parameters()) +
    list(model.eo_e2.parameters()) + list(model.eo_e3.parameters()) +
    list(model.eo_e4.parameters()) +
    list(model.sar_e0.parameters()) + list(model.sar_e1.parameters()) +
    list(model.sar_e2.parameters()) + list(model.sar_e3.parameters()) +
    list(model.sar_e4.parameters())
)

# Decoder + CBAM blocks + head → high LR (randomly initialized)
decoder_params = (
    list(model.bottleneck_cbam.parameters()) +
    list(model.d3.parameters()) + list(model.d2.parameters()) +
    list(model.d1.parameters()) + list(model.d0.parameters()) +
    list(model.up_final.parameters()) + list(model.head.parameters())
)

optimizer = torch.optim.AdamW(
    [
        {"params": encoder_params, "lr": ENCODER_LR},
        {"params": decoder_params, "lr": DECODER_LR},
    ],
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=T_MAX, eta_min=ETA_MIN
)

best_iou   = 0.0
no_improve = 0

print(f"Encoder params: {sum(p.numel() for p in encoder_params):,}")
print(f"Decoder params: {sum(p.numel() for p in decoder_params):,}")
print(f"LR: encoder={ENCODER_LR} | decoder={DECODER_LR}")
print("Optimizer ready ✓")

In [ ]:
# ── Cell 7 — Metrics + Train/Eval Loops ──────────────────────────────────────

def compute_metrics(pred_logits, true_mask, threshold=THRESHOLD):
    pred   = (torch.sigmoid(pred_logits) > threshold).float().view(-1)
    target = true_mask.float().view(-1)
    eps    = 1e-8

    tp = ((pred == 1) & (target == 1)).sum().float()
    fp = ((pred == 1) & (target == 0)).sum().float()
    fn = ((pred == 0) & (target == 1)).sum().float()
    tn = ((pred == 0) & (target == 0)).sum().float()

    return {
        "tp": tp.item(), "fp": fp.item(),
        "fn": fn.item(), "tn": tn.item(),
        "iou":       (tp / (tp + fp + fn + eps)).item(),
        "precision": (tp / (tp + fp + eps)).item(),
        "recall":    (tp / (tp + fn + eps)).item(),
        "f1":        (2*tp / (2*tp + fp + fn + eps)).item(),
    }


def compute_epoch_metrics(tp, fp, fn, tn):
    eps = 1e-8
    tp, fp, fn, tn = [torch.tensor(x, dtype=torch.float32) for x in (tp, fp, fn, tn)]
    return {
        "iou":       (tp / (tp + fp + fn + eps)).item(),
        "precision": (tp / (tp + fp + eps)).item(),
        "recall":    (tp / (tp + fn + eps)).item(),
        "f1":        (2*tp / (2*tp + fp + fn + eps)).item(),
    }

def train_one_epoch(model, loader, optimizer, loss_fn, device):
    model.train()
    total_loss, total_pixels = 0.0, 0
    tp = fp = fn = tn = 0.0

    for eo, sar, masks, _ in loader:
        eo    = eo.to(device)
        sar   = sar.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()
        logits = model(eo, sar)  # (B, 1, H, W)
        loss   = loss_fn(logits, masks)
        loss.backward()
        optimizer.step()

        n_px = eo.size(0) * masks.shape[-2] * masks.shape[-1]
        total_loss   += loss.item() * n_px
        total_pixels += n_px

        m = compute_metrics(logits.detach(), masks.detach())
        tp += m["tp"]; fp += m["fp"]; fn += m["fn"]; tn += m["tn"]

    return {
        "loss": total_loss / max(total_pixels, 1),
        "tp": tp, "fp": fp, "fn": fn, "tn": tn,
    }

def evaluate_one_epoch(model, loader, loss_fn, device):
    model.eval()
    total_loss, total_pixels = 0.0, 0
    tp = fp = fn = tn = 0.0

    with torch.no_grad():
        for eo, sar, masks, _ in loader:
            eo    = eo.to(device)
            sar   = sar.to(device)
            masks = masks.to(device)

            logits = model(eo, sar)
            loss   = loss_fn(logits, masks)

            n_px         = eo.size(0) * masks.shape[-2] * masks.shape[-1]
            total_loss  += loss.item() * n_px
            total_pixels += n_px

            m   = compute_metrics(logits, masks)
            tp += m["tp"];  fp += m["fp"];  fn += m["fn"];  tn += m["tn"]

    return {"loss": total_loss / max(total_pixels, 1),
            "tp": tp, "fp": fp, "fn": fn, "tn": tn}


# Quick sanity
m_p = compute_metrics(torch.full((1,1,4,4), 4.0), torch.ones(1,4,4))
m_w = compute_metrics(torch.full((1,1,4,4), 4.0), torch.zeros(1,4,4))
print(f"Perfect → IoU: {m_p['iou']:.4f} | F1: {m_p['f1']:.4f}")
print(f"Wrong   → IoU: {m_w['iou']:.4f} | F1: {m_w['f1']:.4f}")
print("Metrics + loops ready ✓")

In [ ]:
# ── Cell 8 — Training Driver ──────────────────────────────────────────────────

train_history = []
val_history   = []

for epoch in range(1, NUM_EPOCHS + 1):

    train_stats = train_one_epoch(model, train_loader, optimizer, loss_fn, DEVICE)
    val_stats     = evaluate_one_epoch(model, val_loader, loss_fn, DEVICE)

    train_m = compute_epoch_metrics(
        train_stats["tp"], train_stats["fp"],
        train_stats["fn"], train_stats["tn"]
    )
    val_m   = compute_epoch_metrics(
        val_stats["tp"], val_stats["fp"],
        val_stats["fn"], val_stats["tn"]
    )

    scheduler.step()

    train_history.append({"loss": train_stats["loss"], **train_m})
    val_history.append(  {"loss": val_stats["loss"],   **val_m})

    lrs = [g["lr"] for g in optimizer.param_groups]

    print(
        f"Epoch {epoch:03d}/{NUM_EPOCHS} | "
        f"TLoss: {train_stats['loss']:.4f} | VLoss: {val_stats['loss']:.4f} | "
        f"TIoU: {train_m['iou']:.4f} | VIoU: {val_m['iou']:.4f} | "
        f"LR enc: {lrs[0]:.2e} dec: {lrs[1]:.2e}"
    )

    val_iou  = val_m["iou"]
    improved = val_iou > best_iou + 1e-5

    if improved:
        best_iou   = val_iou
        no_improve = 0
        torch.save(model.state_dict(), CHECKPOINT)
        print(f"  ✓ Saved best (Val IoU: {best_iou:.4f})")
    else:
        no_improve += 1
        print(f"  No improvement {no_improve}/{PATIENCE}")

    if no_improve >= PATIENCE:
        print(f"Early stopping at epoch {epoch}")
        break

print(f"\nTraining complete. Best Val IoU: {best_iou:.4f}")

In [ ]:
# ── Cell 9 — Final Evaluation (TTA) ──────────────────────────────────────────

best_model = DualEncoderUNet(IN_CH_EO, IN_CH_SAR, NUM_CLASSES).to(DEVICE)
best_model.load_state_dict(
    torch.load(CHECKPOINT, map_location=DEVICE, weights_only=True)
)
best_model.eval()


def tta_evaluate(model, loader, loss_fn, device, threshold = 0.4):
    """4-flip TTA: original + hflip + vflip + hvflip, average probabilities."""
    model.eval()
    total_loss, total_pixels = 0.0, 0
    tp = fp = fn = tn = 0.0

    with torch.no_grad():
        for eo, sar, masks, _ in loader:
            eo    = eo.to(device)
            sar   = sar.to(device)
            masks = masks.to(device)

            def fwd(eo_t, sar_t):
                return torch.sigmoid(model(eo_t, sar_t))

            p0  = fwd(eo, sar)
            p1  = torch.flip(fwd(torch.flip(eo,[3]), torch.flip(sar,[3])), [3])
            p2  = torch.flip(fwd(torch.flip(eo,[2]), torch.flip(sar,[2])), [2])
            p3  = torch.flip(fwd(torch.flip(eo,[2,3]), torch.flip(sar,[2,3])), [2,3])

            avg_prob  = (p0 + p1 + p2 + p3) / 4.0
            avg_logit = torch.log(avg_prob / (1 - avg_prob + 1e-8) + 1e-8)

            loss = loss_fn(avg_logit, masks)

            n_px         = eo.size(0) * masks.shape[-2] * masks.shape[-1]
            total_loss  += loss.item() * n_px
            total_pixels += n_px

            m = compute_metrics(avg_logit, masks, threshold=threshold)
            tp += m["tp"]; fp += m["fp"]; fn += m["fn"]; tn += m["tn"]
        
    return {"loss": total_loss / max(total_pixels, 1),
            "tp": tp, "fp": fp, "fn": fn, "tn": tn}


for split, loader in [("VALIDATION", val_loader), ("TEST", test_loader)]:
    stats   = tta_evaluate(best_model, loader, loss_fn, DEVICE)
    metrics = compute_epoch_metrics(stats["tp"], stats["fp"], stats["fn"], stats["tn"])
    print("=" * 60)
    print(f"{split} RESULTS — {CHECKPOINT} + TTA + threshold={THRESHOLD}")
    print("=" * 60)
    print(f"Loss:      {stats['loss']:.4f}")
    print(f"IoU:       {metrics['iou']:.4f}")
    print(f"Precision: {metrics['precision']:.4f}")
    print(f"Recall:    {metrics['recall']:.4f}")
    print(f"F1:        {metrics['f1']:.4f}")
    print(f"TP: {stats['tp']:.0f} | FP: {stats['fp']:.0f} | "
          f"FN: {stats['fn']:.0f} | TN: {stats['tn']:.0f}")


# ── Threshold sweep on VALIDATION only ───────────────────────────────────────
thresholds = [0.15, 0.20, 0.25, 0.30, 0.35, 0.40]

print("=== Validation threshold sweep ===")
for thr in thresholds:
    stats = tta_evaluate(best_model, val_loader, loss_fn, DEVICE, threshold=thr)
    metrics = compute_epoch_metrics(
        stats["tp"], stats["fp"], stats["fn"], stats["tn"]
    )
    print(
        f"thr={thr:.2f} | "
        f"IoU={metrics['iou']:.4f} | "
        f"P={metrics['precision']:.4f} | "
        f"R={metrics['recall']:.4f} | "
        f"F1={metrics['f1']:.4f}"
    )

In [ ]:
from IPython.display import FileLink
FileLink('best_v9(4b - 8 batches).pth')